# 03 — LSTM Forecasting
Trains the LSTM model per currency pair, tunes hyperparameters, and compares with ARIMA.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from ml.training.train import train
from ml.training.evaluate import compare_models
from ml.models.model_utils import load_model, load_scaler
from ml.models.lstm_model import LSTMForecaster

PAIRS = ['USD_EUR', 'USD_AUD', 'USD_NZD']
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 1. Train LSTM on All Pairs (default hyperparameters)

In [ ]:
lstm_results = []
for pair in PAIRS:
    print(f'\n--- {pair} ---')
    result = train(pair_name=pair, hidden_size=128, num_layers=2,
                   dropout=0.2, lr=1e-3, batch_size=64, epochs=50, patience=10)
    lstm_results.append(result)

compare_models(lstm_results)

## 2. Bayesian Hyperparameter Tuning (one pair as demo)

In [ ]:
from ml.training.hyperparameter_tuning import bayesian_search, load_best_params

# Run on USD_EUR (change pair as needed; each study takes ~10-30 min)
best = bayesian_search('USD_EUR', n_trials=20)
print('Best params found:', best)

## 3. Retrain with Best Params

In [ ]:
tuned_results = []
for pair in PAIRS:
    try:
        params = load_best_params(pair, strategy='bayesian')
        params.pop('strategy', None)
        params.pop('best_val_loss', None)
        result = train(pair_name=pair, epochs=100, patience=15, **params)
        result['model'] = 'LSTM (tuned)'
        tuned_results.append(result)
    except FileNotFoundError:
        print(f'No tuning results for {pair}, skipping.')

compare_models(tuned_results)

## 4. ARIMA Baseline Comparison

In [ ]:
from pathlib import Path
from ml.models.arima_baseline import evaluate_arima

processed_dir = Path('../data/processed')
arima_results = []
for pair in PAIRS:
    result = evaluate_arima(pair, processed_dir, order=(5, 1, 0))
    arima_results.append(result)

all_results = lstm_results + arima_results
compare_models(all_results)

## 5. Forecast vs Actual Plot

In [ ]:
from pathlib import Path
import torch

PROCESSED_DIR = Path('../data/processed')

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, pair in zip(axes, PAIRS):
    X_test = torch.tensor(np.load(PROCESSED_DIR / pair / 'X_test.npy'), dtype=torch.float32)
    y_test = np.load(PROCESSED_DIR / pair / 'y_test.npy')

    input_size = X_test.shape[2]
    model = LSTMForecaster(input_size=input_size)
    model = load_model(model, pair, tag='best')
    model.eval()

    with torch.no_grad():
        preds = model(X_test).numpy().squeeze()

    ax.plot(y_test, label='Actual',    linewidth=1)
    ax.plot(preds,  label='Predicted', linewidth=1, alpha=0.8)
    ax.set_title(f'{pair.replace("_", "/")} — LSTM Forecast vs Actual (Test Set)')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/forecast_vs_actual.png', dpi=150)
plt.show()